In [1]:
import pandas as pd
import geopandas as gpd
from libpysal import graph
from torch_geometric.data import Data
import torch
import numpy as np
import math
import os

In [2]:
os.makedirs('data/pyg', exist_ok=True)

## Load data

In [3]:
input_attribx_path  = 'data/processed/28714514_2021UKCensus_processed_20251009001rs.parquet'
input_spatial_path  = 'data/spatial/OA/combined/uk_oa.gpkg'
dataset_nickname    = 'ukc2021-proc20251009001rs'
output_pygdata_path = f'data/pyg/{dataset_nickname}__pygdata.pth'
output_oa_idx_path  = f'data/pyg/{dataset_nickname}__oa_index.parquet'

In [4]:
input_spatial = gpd.read_file(input_spatial_path)
input_spatial.set_index('OA', inplace=True)
print(f"Spatial index: {input_spatial.index.name}, shape: {input_spatial.shape}")
input_spatial

Spatial index: OA, shape: (239023, 1)


,geometry
OA,
E00000001,"MULTIPOLYGON (((532303.492 181814.11, 532213.3..."
E00000003,"MULTIPOLYGON (((532213.378 181846.192, 532190...."
E00000005,"MULTIPOLYGON (((532180.131 181763.02, 532219.1..."
E00000007,"MULTIPOLYGON (((532201.292 181668.18, 532267.7..."
E00000010,"MULTIPOLYGON (((532127.958 182133.192, 532089...."
...,...
N20003776,"MULTIPOLYGON (((170107.583 504381.049, 170115...."
N20003777,"MULTIPOLYGON (((170071.23 504439.6, 170115.329..."
N20003778,"MULTIPOLYGON (((170178.979 503909.159, 170141...."


In [5]:
input_attribx = pd.read_parquet(input_attribx_path)
print(f"Dataset index: {input_attribx.index.name}, shape: {input_attribx.shape}")
input_attribx

Dataset index: OA, shape: (239023, 171)


,uk001002_ntot_clipd_minmax,uk002002_ntot_noout_minmax,uk002003_ntot_noout_minmax,uk002004_ntot_clipd_minmax,uk002005_ntot_noout_minmax,uk002006_ntot_clipd_minmax,uk003002_ntot_noout_minmax,uk003003_ntot_noout_minmax,uk003005_ntot_noout_minmax,uk003006_ntot_noout_minmax,...,uk030005_ntot_clipd_ihs_minmax,uk030006_ntot_clipd_ihs_minmax,uk046004_ntot_clipd_ihs_minmax,uk046005_ntot_clipd_ihs_minmax,uk046009_ntot_clipd_ihs_minmax,uk046010_ntot_clipd_ihs_minmax,uk061006_ntot_clipd_ihs_minmax,uk062010_ntot_clipd_ihs_minmax,uk066008_ntot_clipd_ihs_minmax,uk066016_ntot_noout_ihs_minmax
OA,,,,,,,,,,,,,,,,,,,,,
E00071085,1.000000,0.305556,0.572115,0.068917,0.215650,0.072985,0.152113,0.611111,0.814389,0.693878,...,0.000000,0.223937,0.427451,0.146423,0.000000,0.000000,0.000000,0.523080,0.478946,0.507327
E00177567,1.000000,0.633985,0.321839,0.050698,0.079320,0.032214,0.264544,0.166667,0.614839,0.666667,...,0.661007,0.356224,0.642472,0.428904,0.000000,0.609488,0.375846,0.509519,0.455178,0.626352
E00039966,1.000000,0.308138,0.661796,0.024641,0.121163,0.020876,0.172855,0.266667,0.827428,0.821918,...,0.000000,0.362560,0.000000,0.384673,0.000000,0.000000,0.000000,0.532128,0.340561,0.581545
E00155752,0.469484,0.400431,0.475673,0.109099,0.202043,0.059420,0.365070,0.222222,0.574783,0.724138,...,0.402945,0.073309,0.211627,0.683376,0.000000,0.307746,0.000000,0.566563,0.237234,0.685577
E00018981,1.000000,0.251574,0.710547,0.060628,0.054204,0.089890,0.175998,0.666667,0.715518,0.747126,...,0.827517,0.782768,0.000000,0.146423,0.699728,0.208073,0.176281,0.581465,0.289491,0.651806
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
E00162823,1.000000,0.193732,0.766900,0.033415,0.089621,0.077851,0.088181,0.700000,0.911963,0.838095,...,0.000000,0.227468,0.260426,0.255916,0.000000,0.000000,0.000000,0.597494,0.550560,0.616580
S00164841,1.000000,0.320912,0.653333,0.000000,0.118300,0.024912,0.221467,0.157895,0.743985,0.861538,...,0.294459,0.115057,0.000000,0.000000,0.000000,0.000000,0.000000,0.306222,0.164451,0.362169
S00152701,1.000000,0.713566,0.207101,0.127232,0.113750,0.017966,0.384123,0.080000,0.601167,0.825000,...,0.000000,0.000000,0.407291,0.536659,0.518283,0.000000,0.293425,0.492251,0.462643,0.661991


In [6]:
# Use the index order from input_attribx
input_spatial = input_spatial.reindex(input_attribx.index)
input_spatial

,geometry
OA,
E00071085,"MULTIPOLYGON (((349786.453 240240.682, 349803...."
E00177567,"MULTIPOLYGON (((535304.333 176387.561, 535338...."
E00039966,"MULTIPOLYGON (((433945.829 396656.961, 433902...."
E00155752,"MULTIPOLYGON (((516523.689 148774.078, 516481...."
E00018981,"MULTIPOLYGON (((539806.299 189888.454, 539900...."
...,...
E00162823,"MULTIPOLYGON (((392464 171730, 392557 171938, ..."
S00164841,"MULTIPOLYGON (((334983.156 667581.809, 334916...."
S00152701,"MULTIPOLYGON (((289651.371 681265.248, 289624...."


In [7]:
# # Use the index order from input_spatial
# # which is more orderly
# uol_info = input_attribx.loc["E00068890"]

# input_attribx = input_attribx.reindex(input_spatial.index)

# uol_info_post_reindex = input_attribx.loc["E00068890"]
# assert (uol_info == uol_info_post_reindex).all(), "Something went wrong with reindexing!"

In [8]:
# Check value of students at Uni of Leicester OA
input_attribx.loc["E00068890", "uk066016_ntot_noout_ihs_minmax"]

np.float64(0.9151241185214207)

## Prepare data

### OA index

In [9]:
input_spatial  = input_spatial.reset_index()
dataset_oa_idx = input_spatial[['OA']].copy()
dataset_oa_idx = dataset_oa_idx.reset_index()
dataset_oa_idx["train_valid_test"] = None
dataset_oa_idx

,index,OA,train_valid_test
0,0,E00071085,None
1,1,E00177567,None
2,2,E00039966,None
3,3,E00155752,None
4,4,E00018981,None
...,...,...,...
239018,239018,E00162823,None
239019,239019,S00164841,None
239020,239020,S00152701,None
239021,239021,E00173525,None


### Attribute data

In [10]:
data_tensor      = torch.tensor(input_attribx.values).float()
data_tensor_nrow = data_tensor.shape[0]
data_tensor_ncol = data_tensor.shape[1]

print(data_tensor.shape)
print(data_tensor)

torch.Size([239023, 171])
tensor([[1.0000, 0.3056, 0.5721,  ..., 0.5231, 0.4789, 0.5073],
        [1.0000, 0.6340, 0.3218,  ..., 0.5095, 0.4552, 0.6264],
        [1.0000, 0.3081, 0.6618,  ..., 0.5321, 0.3406, 0.5815],
        ...,
        [1.0000, 0.7136, 0.2071,  ..., 0.4923, 0.4626, 0.6620],
        [1.0000, 0.8180, 0.1538,  ..., 0.9000, 0.6977, 0.9297],
        [1.0000, 0.2165, 0.7587,  ..., 0.5690, 0.4112, 0.6398]])


### Train, valid and test split

In [11]:
split_size_valid = math.floor(data_tensor_nrow * 0.1)
split_size_test  = math.floor(data_tensor_nrow * 0.1)
split_size_train = data_tensor_nrow - split_size_valid - split_size_test

print(f"Train size: {split_size_train}, Validation size: {split_size_valid}, Test size: {split_size_test}")

Train size: 191219, Validation size: 23902, Test size: 23902


In [12]:
train_dataset, valid_dataset, test_dataset = torch.utils.data.random_split(
    data_tensor,
    [split_size_train, split_size_valid, split_size_test]
)

In [13]:
dataset_oa_idx.loc[list(train_dataset.indices), "train_valid_test"] = "train"
dataset_oa_idx.loc[list(valid_dataset.indices), "train_valid_test"] = "valid"
dataset_oa_idx.loc[list(test_dataset.indices), "train_valid_test"] = "test"
dataset_oa_idx

,index,OA,train_valid_test
0,0,E00071085,train
1,1,E00177567,valid
2,2,E00039966,test
3,3,E00155752,train
4,4,E00018981,train
...,...,...,...
239018,239018,E00162823,train
239019,239019,S00164841,train
239020,239020,S00152701,train
239021,239021,E00173525,train


In [14]:
none_check = dataset_oa_idx["train_valid_test"].isna()
none_count = int(none_check.sum())
print(f"Number of None values: {none_count}")

train_check = dataset_oa_idx["train_valid_test"] == "train"
train_count = int(train_check.sum())
print(f"Number of train samples: {train_count}")

val_check = dataset_oa_idx["train_valid_test"] == "valid"
val_count = int(val_check.sum())
print(f"Number of valid samples: {val_count}")

test_check = dataset_oa_idx["train_valid_test"] == "test"
test_count = int(test_check.sum())
print(f"Number of test samples: {test_count}")

Number of None values: 0
Number of train samples: 191219
Number of valid samples: 23902
Number of test samples: 23902


### Spatial graph

In [15]:
g_queen = graph.Graph.build_contiguity(input_spatial, rook=False)

In [16]:
input_spatial_centroids = input_spatial.copy()
input_spatial_centroids["geometry"] = input_spatial_centroids["geometry"].centroid
g_knn8 = graph.Graph.build_knn(input_spatial_centroids, k=8)
del input_spatial_centroids

In [17]:
g_queen_knn8 = graph.Graph.union(g_queen, g_knn8)

### Spatial graph to PyG format

[PyG message passing propagate](https://pytorch-geometric.readthedocs.io/en/2.7.0/generated/torch_geometric.nn.conv.MessagePassing.html#torch_geometric.nn.conv.MessagePassing.propagate):

- from nodes in `edge_index[0]` are sent 
- to nodes in `edge_index[1]`

In [18]:
adj_queen_knn8 = g_queen_knn8.adjacency

focal_idx    = adj_queen_knn8.index.get_level_values(0).to_numpy()
neighbor_idx = adj_queen_knn8.index.get_level_values(1).to_numpy()

edge_index_np = np.stack([
        # source_to_target: 
        # messages go neighbour -> focal
        neighbor_idx, focal_idx
    ], axis=0)
edge_index = torch.from_numpy(edge_index_np).long()

### PyG Data object

In [19]:
train_mask = torch.zeros(data_tensor.shape[0], dtype=torch.bool)
train_mask[train_dataset.indices] = True

valid_mask = torch.zeros(data_tensor.shape[0], dtype=torch.bool)
valid_mask[valid_dataset.indices] = True

test_mask = torch.zeros(data_tensor.shape[0], dtype=torch.bool)
test_mask[test_dataset.indices] = True

In [20]:
pyg_data = Data(
    x=data_tensor,
    edge_index=edge_index,
    train_mask=train_mask,
    valid_mask=valid_mask,
    test_mask=test_mask,
)

In [21]:
print(pyg_data)
print(f"Graph data splits\n Train: {pyg_data.train_mask.sum()}\n Valid: {pyg_data.valid_mask.sum()}\n Test: {pyg_data.test_mask.sum()}\n\n")

Data(x=[239023, 171], edge_index=[2, 2212476], train_mask=[239023], valid_mask=[239023], test_mask=[239023])
Graph data splits
 Train: 191219
 Valid: 23902
 Test: 23902




## Check

In [22]:
odd_pyg_edges = pyg_data.edge_index[:, pyg_data.edge_index[1] == dataset_oa_idx[dataset_oa_idx["OA"]=='E00001096']["index"].values[0]]
odd_pyg_edges

tensor([[ 18726,  44791,  73933, 101086, 121724, 138501, 161233, 202326],
        [ 45070,  45070,  45070,  45070,  45070,  45070,  45070,  45070]])

In [23]:
odd_neigh = dataset_oa_idx[dataset_oa_idx["index"].isin(odd_pyg_edges[0].numpy())].copy()
input_spatial.merge(odd_neigh, left_on="OA", right_on="OA", how="inner").explore()

## Save

In [24]:
torch.save(pyg_data, output_pygdata_path)

# # To load
# data = torch.load('...', weights_only=False)

In [25]:
dataset_oa_idx.to_parquet(output_oa_idx_path)
dataset_oa_idx.to_csv(output_oa_idx_path.replace('.parquet', '.csv'), index=False)